# Importing Necessary Libraries:

In [2]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Conv1D, Dense, LSTM, GRU, TextVectorization, Embedding
from tensorflow.keras.layers import GlobalAveragePooling1D, GlobalMaxPool1D, Bidirectional
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

import tensorflow_hub as hub
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Downloading Helper Functions Script

In [3]:
# Run the below lines if helper functions are not already downloaded:
# Downloading Helper Functions:

#import wget
#url = 'https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py'
#file = wget.download(url)

In [4]:
from helper_functions import calculate_results

## Imprting The Data:

In [5]:
# Getting Data From Github:
!git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git

fatal: destination path 'pubmed-rct' already exists and is not an empty directory.


In [6]:
# Data Directory:
data_dir = 'pubmed-rct/PubMed_200k_RCT_numbers_replaced_with_at_sign/'

In [7]:
# Checking Contents of Data Directory:
os.listdir(data_dir)

['test.txt', 'dev.txt', 'train.zip']

## Creating Functions to Pre-Process The Data

In [8]:
# Function to Read Lines from Files in Data Directory:
def get_lines(filename):
    """
      Reads filename (a text file) and returns the lines of text as a list.

      Args:
          filename: a string containing the target filepath to read.

      Returns:
          A list of strings with one string per line from the target filename.
          For example:
          ["this is the first line of filename",
           "this is the second line of filename",
           "..."]
    """

    with open(filename, mode= 'r') as f:
        return f.readlines()

In [10]:
train_lines = get_lines(data_dir + 'train.txt')
train_lines[:20]

['###24491034\n',
 'BACKGROUND\tThe emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .\n',
 'BACKGROUND\tThis paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .\n',
 'METHODS\tThis study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .\n',
 "METHODS\tThe intervention group will participate in the online group program ` Positive Outlook ' .\n",
 'METHODS\tThe program is based on self-efficacy theory and uses a self-management approach to enhance skills , confidence and abilities to manage the psychosocial issues associated with HIV in daily life .\n',
 'METHODS\tParticipants will access the program f

In [11]:
# Function to Pre-Process The Data in Train, Test and Validation Files:

def preprocess_text_with_line_numbers(filename):
    """Returns a list of dictionaries of abstract line data.

      Takes in filename, reads its contents and sorts through each line,
      extracting things like the target label, the text of the sentence,
      how many sentences are in the current abstract and what sentence number
      the target line is.

      Args:
          filename: a string of the target text file to read and extract line data
          from.

      Returns:
          A list of dictionaries each containing a line from an abstract,
          the lines label, the lines position in the abstract and the total number
          of lines in the abstract where the line is from. For example:

          [{"target": 'CONCLUSION',
            "text": The study couldn't have gone better, turns out people are kinder than you think",
            "line_number": 8,
            "total_lines": 8}]
    """

    input_data = get_lines(filename)
    abstract_lines = ""
    abstract_samples = []

    for line in input_data:

        if line.startswith("###"):
          abstract_id = line
          abstract_lines = ""

        elif line.isspace():
            abstract_line_split = abstract_lines.splitlines()

            for abstract_line_number, abstract_line in enumerate(abstract_line_split):
                line_data = {}
                target_text_split = abstract_line.split("\t")
                line_data["Target"] = target_text_split[0]
                line_data["Text"] = target_text_split[1]
                line_data["Line_Number"] = abstract_line_number
                line_data["Total_Lines"] = len(abstract_line_split) - 1
                abstract_samples.append(line_data)

        else:
            abstract_lines += line

    return abstract_samples

In [12]:
# Getting All File Names from Data Directory in a list:
filenames = [data_dir + filename for filename in os.listdir(data_dir)]

In [13]:
filenames

['pubmed-rct/PubMed_200k_RCT_numbers_replaced_with_at_sign/test.txt',
 'pubmed-rct/PubMed_200k_RCT_numbers_replaced_with_at_sign/dev.txt',
 'pubmed-rct/PubMed_200k_RCT_numbers_replaced_with_at_sign/train.zip',
 'pubmed-rct/PubMed_200k_RCT_numbers_replaced_with_at_sign/train.txt']

In [14]:
# Getting Data from All files:
train_data = preprocess_text_with_line_numbers(filename= filenames[3])
val_data = preprocess_text_with_line_numbers(filename= filenames[1])
test_data = preprocess_text_with_line_numbers(filename= filenames[0])

In [15]:
train_data[:5]

[{'Target': 'BACKGROUND',
  'Text': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .',
  'Line_Number': 0,
  'Total_Lines': 10},
 {'Target': 'BACKGROUND',
  'Text': 'This paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .',
  'Line_Number': 1,
  'Total_Lines': 10},
 {'Target': 'METHODS',
  'Text': 'This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .',
  'Line_Number': 2,
  'Total_Lines': 10},
 {'Target': 'METHODS',
  'Text': "The intervention group will participate in the online group program ` Positive Outlook ' .",
  'Line_Number': 3,
  'Total_Lines': 10},
 {'Target': 'METHODS',
  '

In [16]:
val_data[:5]

[{'Target': 'BACKGROUND',
  'Text': 'Adrenergic activation is thought to be an important determinant of outcome in subjects with chronic heart failure ( CHF ) , but baseline or serial changes in adrenergic activity have not been previously investigated in a large patient sample treated with a powerful antiadrenergic agent .',
  'Line_Number': 0,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'Systemic venous norepinephrine was measured at baseline , @ months , and @ months in the beta-Blocker Evaluation of Survival Trial ( BEST ) , which compared placebo treatment with the beta-blocker/sympatholytic agent bucindolol .',
  'Line_Number': 1,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'Baseline norepinephrine level was associated with a progressive increase in rates of death or death plus CHF hospitalization that was independent of treatment group .',
  'Line_Number': 2,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'On multivariate analysis , baseline norepineph

In [17]:
test_data[:5]

[{'Target': 'BACKGROUND',
  'Text': 'Many pathogenic processes and diseases are the result of an erroneous activation of the complement cascade and a number of inhibitors of complement have thus been examined for anti-inflammatory actions .',
  'Line_Number': 0,
  'Total_Lines': 10},
 {'Target': 'BACKGROUND',
  'Text': 'It was recently demonstrated that supraphysiological concentrations of the endogenous complement inhibitor MAp@ ( also denoted MAP@ ) protect against myocardial reperfusion injury .',
  'Line_Number': 1,
  'Total_Lines': 10},
 {'Target': 'BACKGROUND',
  'Text': 'In the present study , we examined the association between outcome after acute myocardial infarction ( MI ) and the plasma levels of MAp@ and its related proteins MASP-@ and MASP-@ in patients with first-time MI .',
  'Line_Number': 2,
  'Total_Lines': 10},
 {'Target': 'BACKGROUND',
  'Text': 'In addition , we compared plasma levels of MAp@ , MASP-@ , and MASP-@ in MI patients to levels in a healthy control grou

In [18]:
# Creeating Dataframes for Training, Validation and Test Data:
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
test_df = pd.DataFrame(test_data)

In [19]:
train_df.head(10)

,Target,Text,Line_Number,Total_Lines
0,BACKGROUND,The emergence of HIV as a chronic condition me...,0,10
1,BACKGROUND,This paper describes the design and evaluation...,1,10
2,METHODS,This study is designed as a randomised control...,2,10
3,METHODS,The intervention group will participate in the...,3,10
4,METHODS,The program is based on self-efficacy theory a...,4,10
5,METHODS,Participants will access the program for a min...,5,10
6,METHODS,Primary outcomes are domain specific self-effi...,6,10
7,METHODS,"Secondary outcomes include : depression , anxi...",7,10
8,METHODS,"Data collection will take place at baseline , ...",8,10
9,CONCLUSIONS,Results of the Positive Outlook study will pro...,9,10


In [20]:
train_df.shape

(2211861, 4)

## Getting Data Ready for The Model

In [21]:
# Converting Text to List of Sentencts for Each Datasets:

train_sentences = train_df['Text'].to_list()
val_sentences = val_df['Text'].to_list()
test_sentences = test_df['Text'].to_list()

In [22]:
train_df.shape, val_df.shape, test_df.shape

((2211861, 4), (28932, 4), (29493, 4))

In [23]:
# Creating Labels as Numerics Labels (Both One-Hot and Label Encoded):
one_hot_encoder = OneHotEncoder(sparse_output= False)
train_labels_one_hot = one_hot_encoder.fit_transform(train_df['Target'].to_numpy().reshape(-1,1))
val_labels_one_hot = one_hot_encoder.fit_transform(val_df['Target'].to_numpy().reshape(-1,1))
test_labels_one_hot = one_hot_encoder.fit_transform(test_df['Target'].to_numpy().reshape(-1,1))

In [24]:
label_encoder = LabelEncoder()

train_labels_encoded = label_encoder.fit_transform(train_df['Target'].to_numpy())
val_labels_encoded = label_encoder.fit_transform(val_df['Target'].to_numpy())
test_labels_encoded = label_encoder.fit_transform(test_df['Target'].to_numpy())

In [25]:
train_labels_one_hot, val_labels_one_hot, test_labels_one_hot

(array([[1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        ...,
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1.]]),
 array([[1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1.],
        ...,
        [0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.]]),
 array([[1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        ...,
        [0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.]]))

In [26]:
train_labels_one_hot.shape , val_labels_one_hot.shape, test_labels_one_hot.shape

((2211861, 5), (28932, 5), (29493, 5))

In [27]:
train_labels_encoded, val_labels_encoded, test_labels_encoded

(array([0, 0, 2, ..., 4, 4, 4]),
 array([0, 4, 4, ..., 4, 1, 1]),
 array([0, 0, 0, ..., 4, 1, 1]))

In [28]:
train_labels_encoded.shape, val_labels_encoded.shape, test_labels_encoded.shape

((2211861,), (28932,), (29493,))

In [30]:
# Converting Line Number and Total Lines to One hot Encoded Features:

train_line_numbers_one_hot = tf.one_hot(train_df["Line_Number"].to_numpy(), depth= 15)
val_line_numbers_one_hot = tf.one_hot(val_df["Line_Number"].to_numpy(), depth= 15)
test_line_numbers_one_hot = tf.one_hot(test_df["Line_Number"].to_numpy(), depth= 15)

train_total_lines_one_hot = tf.one_hot(train_df["Total_Lines"].to_numpy(), depth= 20)
val_total_lines_one_hot = tf.one_hot(val_df["Total_Lines"].to_numpy(), depth= 20)
test_total_lines_one_hot = tf.one_hot(test_df["Total_Lines"].to_numpy(), depth= 20)

In [31]:
train_line_numbers_one_hot

<tf.Tensor: shape=(2211861, 15), dtype=float32, numpy=
array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

In [32]:
train_total_lines_one_hot

<tf.Tensor: shape=(2211861, 20), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

# Embeddings and Model Creation:

## Token (Word) Embedding

### Pre-Trained Token Embedding

In [33]:
# Importing Pre-Trained Embedding from Tensorflow Hub for Token (Word) Embedding:
tf_hub_token_embedding_layer = hub.KerasLayer(handle= 'https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/universal-sentence-encoder/2',
                                             trainable= False,
                                             name= 'Universal_Sentence_Encoder')

### Custom Token Embedding

In [37]:
# Finding Average Number of Words in Training Sentences:
sent_lens = [len(sentence.split()) for sentence in train_sentences]
avg_sent_len = np.mean(sent_lens)
avg_sent_len

26.229355280462922

In [38]:
# Length of Sentence That Covers 95% of training Data:
np.percentile(sent_lens, 95)

54.0

In [39]:
Max_Tokens_Word = 68000 # How many words are in our vocabulary? (taken from 3.2 in https://arxiv.org/pdf/1710.06071.pdf)
output_seq_len_word = 55

In [40]:
# Creartin Token Vectorizer:
token_vectorizer = TextVectorization(max_tokens= Max_Tokens_Word,
                                    output_sequence_length= output_seq_len_word,
                                    name= 'Token_Vectorizer')

In [41]:
# Adapting Token Vector to Training Words:
token_vectorizer.adapt(train_sentences)

In [42]:
# How Many Words are there in Token Vectorizer Vocabulatry:
token_vect_vocab = token_vectorizer.get_vocabulary()
len(token_vect_vocab)

68000

In [43]:
# Creating Custome Token (Word) Embedding:

token_embed = Embedding(input_dim= len(token_vect_vocab),
                       output_dim= 128,
                       name= 'Token_Embedding')